In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
gdf_hex = gpd.read_file("../data/processed/barcelona_hex_grid.geojson")
df_income = pd.read_csv("../data/raw/barcelona_income.csv", sep=None, engine="python")

print(f"Hex grid: {gdf_hex.shape}")
print(f"Income data: {df_income.shape}")
print(f"\nHex columns: {gdf_hex.columns.tolist()}")
print(f"\nIncome columns: {df_income.columns.tolist()}")

Hex grid: (1620, 8)
Income data: (1068, 7)

Hex columns: ['h3_index', 'catchment_store_count', 'catchment_type_diversity', 'is_food_desert', 'store_count', 'type_diversity', 'barri_name', 'geometry']

Income columns: ['Any', 'Codi_Districte', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'Seccio_Censal', 'Import_Euros']


In [8]:
gdf_barris = gpd.read_file("../data/raw/barcelona_barris.geojson").to_crs("EPSG:4326")

# NOM is the actual neighbourhood name column
print(gdf_barris[["BARRI", "NOM"]].head(10).to_string())

# Drop the wrong barri_name column (numeric codes) and re-join using NOM
gdf_hex = gdf_hex.drop(columns=["barri_name", "index_right"], errors="ignore")

gdf_hex = gpd.sjoin(
    gdf_hex,
    gdf_barris[["geometry", "NOM"]].rename(columns={"NOM": "barri_name"}),
    how="left",
    predicate="intersects"
).drop(columns=["index_right"], errors="ignore")

# Keep one row per hex (sjoin can duplicate if hex intersects multiple barris)
gdf_hex = gdf_hex.drop_duplicates(subset="h3_index", keep="first")

print(f"\nHexes with correct barri name: {gdf_hex['barri_name'].notna().sum()} / {len(gdf_hex)}")
print(gdf_hex["barri_name"].dropna().unique()[:10])

  BARRI                              NOM
0    01                         el Raval
1    02                   el Barri Gòtic
2    03                   la Barceloneta
3    07           la Dreta de l'Eixample
4    08  l'Antiga Esquerra de l'Eixample
5    09   la Nova Esquerra de l'Eixample
6    10                      Sant Antoni
7    11                     el Poble-sec
8    12       la Marina del Prat Vermell
9    13                la Marina de Port

Hexes with correct barri name: 997 / 997
['les Tres Torres' 'Vallvidrera, el Tibidabo i les Planes'
 'la Barceloneta' "la Dreta de l'Eixample" 'el Poble-sec'
 'Sant Gervasi - la Bonanova' 'Sants' 'la Font de la Guatlla' 'el Carmel'
 'la Marina del Prat Vermell']


In [3]:
# Print income data so we can identify the right columns
print(df_income.head(10).to_string())
print(f"\nUnique values in first column: {df_income.iloc[:, 0].unique()[:10]}")

    Any  Codi_Districte Nom_Districte  Codi_Barri Nom_Barri  Seccio_Censal  Import_Euros
0  2022               1  Ciutat Vella           1  el Raval              1         15940
1  2022               1  Ciutat Vella           1  el Raval              2         13841
2  2022               1  Ciutat Vella           1  el Raval              3         12732
3  2022               1  Ciutat Vella           1  el Raval              4         15749
4  2022               1  Ciutat Vella           1  el Raval              5         13190
5  2022               1  Ciutat Vella           1  el Raval              6         14262
6  2022               1  Ciutat Vella           1  el Raval              7         13335
7  2022               1  Ciutat Vella           1  el Raval              8         10692
8  2022               1  Ciutat Vella           1  el Raval              9         13723
9  2022               1  Ciutat Vella           1  el Raval             10         11833

Unique values in fir

In [9]:
from difflib import get_close_matches

BARRI_COL  = "Nom_Barri"
INCOME_COL = "Import_Euros"

# Aggregate to barri level
df_income_barri = (
    df_income.groupby(BARRI_COL)[INCOME_COL]
    .median()
    .reset_index()
    .rename(columns={BARRI_COL: "barri_name", INCOME_COL: "income_index"})
)

# ── Debug: see what names look like on both sides ─────────────────
hex_barri_names    = sorted(gdf_hex["barri_name"].dropna().unique().tolist())
income_barri_names = sorted(df_income_barri["barri_name"].tolist())

print("── Sample HEX barri names ──")
print(hex_barri_names[:15])
print("\n── Sample INCOME barri names ──")
print(income_barri_names[:15])

# ── Fuzzy match: map each income name to closest hex name ─────────
def normalise(s):
    return s.strip().lower()

hex_norm_map = {normalise(n): n for n in hex_barri_names}

matched_pairs = {}
unmatched = []

for name in income_barri_names:
    norm = normalise(name)
    if norm in hex_norm_map:
        matched_pairs[name] = hex_norm_map[norm]
    else:
        close = get_close_matches(norm, hex_norm_map.keys(), n=1, cutoff=0.7)
        if close:
            matched_pairs[name] = hex_norm_map[close[0]]
        else:
            unmatched.append(name)

print(f"\n✓ Matched: {len(matched_pairs)} / {len(income_barri_names)} barris")
print(f"✗ Unmatched: {unmatched}")

# Apply mapping and merge
# Drop stale income column from previous run before merging
gdf_hex = gdf_hex.drop(columns=["income_index", "barri_name_hex", 
                                  "income_index_x", "income_index_y"], errors="ignore")

df_income_barri["barri_name_hex"] = df_income_barri["barri_name"].map(matched_pairs)

gdf_hex = gdf_hex.merge(
    df_income_barri[["barri_name_hex", "income_index"]].dropna(),
    left_on="barri_name",
    right_on="barri_name_hex",
    how="left"
).drop(columns=["barri_name_hex"], errors="ignore")

matched = gdf_hex["income_index"].notna().sum()
print(f"\nHexes with income matched: {matched} / {len(gdf_hex)} ({matched/len(gdf_hex)*100:.1f}%)")

── Sample HEX barri names ──
['Baró de Viver', 'Can Baró', 'Can Peguera', 'Canyelles', 'Ciutat Meridiana', 'Diagonal Mar i el Front Marítim del Poblenou', 'Horta', 'Hostafrancs', 'Montbau', 'Navas', 'Pedralbes', 'Porta', 'Provençals del Poblenou', 'Sant Andreu', 'Sant Antoni']

── Sample INCOME barri names ──
['Baró de Viver', 'Can Baró', 'Can Peguera', 'Canyelles', 'Ciutat Meridiana', 'Diagonal Mar i el Front Marítim del Poblenou', 'Horta', 'Hostafrancs', 'Montbau', 'Navas', 'Pedralbes', 'Porta', 'Provençals del Poblenou', 'Sant Andreu', 'Sant Antoni']

✓ Matched: 71 / 73 barris
✗ Unmatched: ['el Congrés i els Indians', 'el Poble Sec - AEI Parc Montjuïc']

Hexes with income matched: 961 / 997 (96.4%)


In [10]:
# Manual name mapping for the 2 unmatched barris
manual_fixes = {
    "el Poble Sec - AEI Parc Montjuïc": "el Poble-sec",
    "el Congrés i els Indians":          "el Congrés i els Indians"  # check exact hex name below
}

# Find the exact name used in gdf_hex for Congrés
congres_match = [n for n in gdf_hex["barri_name"].dropna().unique() if "congr" in n.lower()]
print(f"Congrés name in hex grid: {congres_match}")

# Apply fixes to income data and re-merge the 2 missing barris
income_fixes = df_income_barri[df_income_barri["barri_name"].isin(manual_fixes.keys())].copy()
income_fixes["barri_name"] = income_fixes["barri_name"].map(manual_fixes)

# Patch income_index for unmatched hexes
for _, row in income_fixes.iterrows():
    mask = (gdf_hex["barri_name"] == row["barri_name"]) & (gdf_hex["income_index"].isna())
    gdf_hex.loc[mask, "income_index"] = row["income_index"]

matched = gdf_hex["income_index"].notna().sum()
print(f"Hexes with income matched after fix: {matched} / {len(gdf_hex)} ({matched/len(gdf_hex)*100:.1f}%)")

Congrés name in hex grid: []
Hexes with income matched after fix: 997 / 997 (100.0%)


In [11]:
def min_max_scale(series):
    """Scale a series to 0–1, handling edge cases."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mn) / (mx - mn)

# ── Component 1: Store Density (40%) ─────────────────────────────
# Catchment store count scaled 0–1
gdf_hex["c1_density"] = min_max_scale(gdf_hex["catchment_store_count"])

# ── Component 2: Store Diversity (20%) ────────────────────────────
# Number of unique shop types in catchment
gdf_hex["c2_diversity"] = min_max_scale(gdf_hex["catchment_type_diversity"])

# ── Component 3: Population Pressure (20%) ────────────────────────
# We use income_index as a proxy (lower income = higher pressure)
# Will be refined if population data is available
# For now: inverted income index as deprivation signal
gdf_hex["c3_pressure"] = 1 - min_max_scale(
    gdf_hex["income_index"].fillna(gdf_hex["income_index"].median())
)

# ── Component 4: Income Deprivation (20%) ─────────────────────────
# Direct deprivation: lower income = higher deprivation = lower access score
gdf_hex["c4_deprivation"] = min_max_scale(
    gdf_hex["income_index"].fillna(gdf_hex["income_index"].median())
)

# ── Composite Score ───────────────────────────────────────────────
WEIGHTS = {"c1_density": 0.40, "c2_diversity": 0.20, 
           "c3_pressure": 0.20, "c4_deprivation": 0.20}

gdf_hex["food_access_score"] = sum(
    gdf_hex[col] * weight for col, weight in WEIGHTS.items()
) * 100  # scale to 0–100

print("Score distribution:")
print(gdf_hex["food_access_score"].describe().round(2))
print(f"\nFood desert hexes (score < 20): {(gdf_hex['food_access_score'] < 20).sum()}")
print(f"Well-served hexes (score > 70): {(gdf_hex['food_access_score'] > 70).sum()}")

Score distribution:
count    997.00
mean      36.92
std       14.67
min       20.00
25%       23.47
50%       35.63
75%       46.94
max       80.00
Name: food_access_score, dtype: float64

Food desert hexes (score < 20): 0
Well-served hexes (score > 70): 17


In [12]:
barri_scores = (
    gdf_hex.groupby("barri_name")
    .agg(
        avg_food_access_score = ("food_access_score", "mean"),
        median_income         = ("income_index", "median"),
        total_stores          = ("store_count", "sum"),
        food_desert_hexes     = ("is_food_desert", "sum"),
        total_hexes           = ("h3_index", "count")
    )
    .reset_index()
)

barri_scores["pct_food_desert"] = (
    barri_scores["food_desert_hexes"] / barri_scores["total_hexes"] * 100
).round(1)

barri_scores = barri_scores.sort_values("avg_food_access_score")

print("🔴 10 most underserved neighbourhoods:")
print(barri_scores.head(10)[["barri_name", "avg_food_access_score", 
                               "pct_food_desert", "total_stores"]].to_string())
print("\n🟢 10 best-served neighbourhoods:")
print(barri_scores.tail(10)[["barri_name", "avg_food_access_score", 
                               "pct_food_desert", "total_stores"]].to_string())

🔴 10 most underserved neighbourhoods:
                               barri_name  avg_food_access_score  pct_food_desert  total_stores
26  Vallvidrera, el Tibidabo i les Planes              21.334191             71.4             2
15               Sant Genís dels Agudells              22.840841             73.3             0
55             la Marina del Prat Vermell              23.315015             59.3            24
8                                 Montbau              23.788789             50.0             1
23                             Torre Baró              25.455750             11.8             2
24                               Vallbona              25.825826              0.0             0
6                                   Horta              26.126126             57.1             6
64                      la Trinitat Vella              26.493994              0.0             2
63                       la Trinitat Nova              26.610360              0.0             2
4 

In [13]:
# Only use hexes where we have income data
df_corr = gdf_hex[["food_access_score", "income_index"]].dropna()

pearson_r,  pearson_p  = stats.pearsonr(df_corr["income_index"], df_corr["food_access_score"])
spearman_r, spearman_p = stats.spearmanr(df_corr["income_index"], df_corr["food_access_score"])

print("── Correlation: Household Income vs Food Access Score ──")
print(f"Pearson  r = {pearson_r:.3f}  (p = {pearson_p:.4f})")
print(f"Spearman r = {spearman_r:.3f}  (p = {spearman_p:.4f})")

if pearson_p < 0.05:
    direction = "positive" if pearson_r > 0 else "negative"
    print(f"\n✓ Statistically significant {direction} correlation (p < 0.05)")
    print(f"  → {'Higher income areas have better food access' if pearson_r > 0 else 'Lower income areas have worse food access'}")

── Correlation: Household Income vs Food Access Score ──
Pearson  r = 0.012  (p = 0.7116)
Spearman r = 0.095  (p = 0.0026)


In [14]:
# Drop list columns before saving
gdf_hex_scored = gdf_hex.drop(
    columns=["shop_types", "store_names", "barri_name_join", "barri_name_income"], 
    errors="ignore"
)

gdf_hex_scored.to_file("../data/processed/barcelona_hex_scored.geojson", driver="GeoJSON")
barri_scores.to_csv("../data/processed/barcelona_barri_scores.csv", index=False)

print(f"Saved hex scored grid: {len(gdf_hex_scored)} hexes")
print(f"Saved barri scores:    {len(barri_scores)} neighbourhoods")

Saved hex scored grid: 997 hexes
Saved barri scores:    72 neighbourhoods
